# Base por urna: perfil do eleitorado × voto sobre os aptos

Uma linha por **urna** (seção principal + agregadas) do Presidente, com o voto
dividido de duas formas: sobre os **válidos** (como na planilha de 2018 do
Sergio Simoni Jr.) e sobre os **eleitores aptos** (PT + bolsonarista + brancos e
nulos + abstenção = 100%), mais a escolaridade do Cadastro Eleitoral somada por urna.

Por que por urna: o voto das seções agregadas sai na seção principal. Casando o
cadastro seção a seção, os eleitores das agregadas ficam sem votos (3,8% das
linhas da planilha de 2018 tinham mais votos válidos que eleitores no perfil).
Com os aptos no denominador isso daria percentuais acima de 100%.

- **Aptos** = `QT_APTOS` do `detalhe_votacao_secao` (na falta dele, a soma de
  `QT_ELEITOR_ELEICAO_FEDERAL` do `eleitorado_local_votacao`, igual em 100% das urnas).
- **`pobres`** = (analfabeto + lê e escreve + fundamental incompleto) ÷ eleitores
  com escolaridade informada × 100, como na planilha.
- A escolaridade do cadastro é a declarada no alistamento e quase nunca é
  atualizada. Os resultados são **associações condicionais** ecológicas, não
  comportamento individual.

Parâmetros: `ANO`, `TURNO` e `UFS` (lista de siglas para teste; `None` = Brasil).
Entrada: zips do TSE em `dados/01_bruto/{ano}/` (ver `03b_baixa_resultados.qmd -P perfil:true`).

In [ ]:
ANO = 2026
TURNO = 2
UFS = None  # ex.: ["PI"] para testar; None = Brasil inteiro

In [ ]:
import re
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ / "src"))

from eleicoes import fontes, urna  # noqa: E402
from eleicoes.config import BRUTO_DIR, NR_BOLSONARO_POR_ANO, NR_PT, PROCESSADO_DIR, RESULTADOS_DIR  # noqa: E402

ANO, TURNO = int(ANO), int(TURNO)
if isinstance(UFS, str):  # -p UFS PI,AC chega como texto
    UFS = re.findall(r"[A-Za-z]{2}", UFS)
UFS = [u.upper() for u in UFS] if UFS else None
NR_BOLSO = NR_BOLSONARO_POR_ANO[ANO]
ANO_PIB = {2018: 2018, 2022: 2022, 2026: 2023}[ANO]  # último PIB municipal publicado em 2026: 2023
SAIDA_DIR = RAIZ / "dados" / "04_analise" / "urna"
SAIDA_DIR.mkdir(parents=True, exist_ok=True)
SUFIXO = f"{ANO}_t{TURNO}" + (f"_{'_'.join(UFS)}" if UFS else "")
# o CSV de uma UF grande passa de vários GB: extrai fora da pasta do OneDrive
TMP_DIR = Path(tempfile.gettempdir()) / "dados_eleicoes_tmp"
CHAVE = ["cd_mun_tse", "nr_zona", "nr_secao"]

## 1. Seções → urnas

Pelo `eleitorado_local_votacao` **do mesmo turno** (o local de uma seção pode mudar
de um turno para o outro). A seção agregada entra na urna da seção principal.

In [ ]:
secoes = urna.le_secao_urna(BRUTO_DIR / str(ANO) / f"eleitorado_local_votacao_{ANO}.zip", TURNO)
if UFS:
    secoes = secoes.loc[secoes["sg_uf"].isin(UFS)]
urnas = urna.monta_urnas(secoes)
print(f"{len(secoes):,} seções → {len(urnas):,} urnas ({(urnas['n_secoes'] > 1).sum():,} com agregadas)")
display(secoes.groupby("tipo_secao").size().rename("seções").to_frame())

## 2. Votos por urna

`fontes.obtem_resultados_secao` lê o `votacao_secao` e, se o `detalhe_votacao_secao`
estiver em disco, traz `qt_aptos` e `qt_comparecimento` do TSE.

In [ ]:
votos = urna.le_votos_urna(ANO, TURNO)
print(f"{len(votos):,} urnas com resultado no votacao_secao")
votos[["qt_pt", "qt_bolso", "qt_outros", "qt_brancos", "qt_nulos", "qt_comparecimento", "qt_aptos"]].sum().to_frame("total")

## 3. Perfil do eleitorado por urna

Um UF por vez (`perfil_eleitor_secao_{ano}_{UF}.zip`), com cache em
`dados/02_processado/perfil/`. A soma por urna junta as agregadas à principal.

In [ ]:
perfil = urna.le_perfil_brasil(ANO, BRUTO_DIR / str(ANO), PROCESSADO_DIR / "perfil", TMP_DIR, UFS)
perfil_urna, perfil_sem_urna = urna.soma_perfil_por_urna(perfil, secoes)
print(f"{len(perfil):,} seções no perfil → {len(perfil_urna):,} urnas")
print(f"seções do perfil que não existem no eleitorado do turno: {len(perfil_sem_urna):,} "
      f"({perfil_sem_urna['qt_eleitores_perfil'].sum():,.0f} eleitores)")

## 4. Junção, região, PIB e indicadores

In [ ]:
base = (
    urnas.merge(votos, on=CHAVE, how="left", validate="one_to_one")
    .merge(perfil_urna, on=CHAVE, how="left", validate="one_to_one")
)
# aptos do detalhe; sem ele (urna sem resultado), a soma do federal do eleitorado
base["qt_aptos"] = base["qt_aptos"].fillna(base["qt_aptos_federal"])

crosswalk = pd.read_parquet(PROCESSADO_DIR / "aux" / "crosswalk_mun_tse_ibge.parquet")
base = urna.adiciona_regiao(base, crosswalk)

pib = urna.le_pib_per_capita(ANO_PIB).set_index("cd_mun")["pib_per_capita"]
base["pib_per_capita"] = base["cd_mun"].map(pib)
quintil_mun = pd.Series(pd.qcut(pib, 5, labels=[1, 2, 3, 4, 5]).astype(float), index=pib.index)
base["quintil_pib_mun"] = base["cd_mun"].map(quintil_mun)

base = urna.calcula_indicadores_urna(base)
base = urna.marca_exclusoes(base)

resumo = base.groupby(base["motivo_exclusao"].replace("", "(entra na análise)")).agg(
    urnas=("id_urna", "size"), aptos=("qt_aptos", "sum"), comparecimento=("qt_comparecimento", "sum")
)
display(resumo)
analise = base.loc[base["motivo_exclusao"] == ""].copy()

## 5. Conferência

In [ ]:
# --- totais iguais aos da etapa 04 (zona × município, que inclui o exterior)
if not UFS:
    ref = pd.read_parquet(RESULTADOS_DIR / f"res_presidente_{ANO}_t{TURNO}_zona_mun.parquet")
    esperado = {
        "qt_pt": ref[f"qt_votos_{NR_PT}"].sum(), "qt_bolso": ref[f"qt_votos_{NR_BOLSO}"].sum(),
        "qt_comparecimento": ref["qt_comparecimento"].sum(),
    }
    for col, valor in esperado.items():
        assert base[col].sum() == valor, (col, base[col].sum(), valor)
    print("totais iguais ao res_presidente_zona_mun:", {k: int(v) for k, v in esperado.items()})

    sem_urna = votos.merge(urnas[CHAVE], on=CHAVE, how="left", indicator=True)["_merge"].eq("left_only").sum()
    assert sem_urna == 0, f"{sem_urna} urnas com voto sem urna no eleitorado"

# --- aptos do detalhe = soma do federal do eleitorado
com_resultado = base["qt_comparecimento"].notna()
igual = (base.loc[com_resultado, "qt_aptos"] == base.loc[com_resultado, "qt_aptos_federal"]).mean()
print(f"aptos do detalhe == soma do federal: {igual:.4%} das urnas com resultado")
assert igual > 0.999

# --- votos fecham o comparecimento
fecha = (base[["qt_pt", "qt_bolso", "qt_outros", "qt_brancos", "qt_nulos"]].sum(axis=1) == base["qt_comparecimento"])[com_resultado]
assert fecha.all(), f"{(~fecha).sum()} urnas em que os votos não fecham o comparecimento"
if TURNO == 2:
    assert (base["qt_outros"].dropna() == 0).all()

# --- as quatro partes sobre os aptos somam 100
partes = analise[["pc_aptos_pt", "pc_aptos_bolso", "pc_aptos_branco_nulo", "pc_aptos_abstencao"]].sum(axis=1)
assert np.allclose(partes, 100), partes.describe()

# --- o perfil conserva o total (nada se perde na soma por urna)
assert perfil_urna["qt_eleitores_perfil"].sum() + perfil_sem_urna["qt_eleitores_perfil"].sum() == perfil["qt_eleitores_perfil"].sum()

# --- o cadastro frente aos aptos
r = analise["razao_cadastro_aptos"]
print(f"perfil/aptos: mediana {r.median():.4f}; entre 0,9 e 1,1 em {r.between(0.9, 1.1).mean():.2%} das urnas; "
      f"acima de 1,1 em {(r > 1.1).sum():,}; abaixo de 0,9 em {(r < 0.9).sum():,}")
display(analise[["pobres", "pc_validos_pt", "pc_aptos_pt", "pc_aptos_bolso", "pc_aptos_branco_nulo", "pc_aptos_abstencao", "pc_facultativo"]].describe().round(2))

## 6. Saída

`dados/04_analise/urna/urna_presidente_{ano}_t{turno}.parquet` (todas as urnas, com
`motivo_exclusao`) e o `.xlsx` no layout da planilha de 2018 (só as urnas que entram
na análise, mais uma aba com as excluídas, o dicionário e as notas).

In [ ]:
base.to_parquet(SAIDA_DIR / f"urna_presidente_{SUFIXO}.parquet", index=False)

ESC = {c: n.upper() for c, n in urna.ROTULOS_ESC.items()}
LAYOUT = [
    ("SG_UF", "sg_uf"), ("CD_MUNICIPIO", "cd_mun_tse"), ("NM_MUNICIPIO", "nm_municipio"),
    ("NR_ZONA", "nr_zona"), ("NR_LOCAL_VOTACAO", "nr_local_votacao"), ("NR_SECAO", "nr_secao"),
    ("N_SECOES", "n_secoes"), ("QT_APTOS", "qt_aptos"), ("QT_COMPARECIMENTO", "qt_comparecimento"),
    ("QT_ABSTENCAO", "qt_abstencao"), ("Total de QT_VOTOS (validos)", "qt_validos"),
    (str(NR_PT), "qt_pt"), ("pt (% validos)", "pc_validos_pt"),
    (str(NR_BOLSO), "qt_bolso"), ("bolso (% validos)", "pc_validos_bolso"),
    ("QT_BRANCOS", "qt_brancos"), ("QT_NULOS", "qt_nulos"),
    ("pt (% aptos)", "pc_aptos_pt"), ("bolso (% aptos)", "pc_aptos_bolso"),
    ("branco+nulo (% aptos)", "pc_aptos_branco_nulo"), ("abstencao (% aptos)", "pc_aptos_abstencao"),
    ("TOTAL", "qt_perfil_esc"),
    *[(ESC[c], f"qt_esc_{c}") for c in urna.ROTULOS_ESC],
    ("pobres", "pobres"), ("razao perfil/aptos", "razao_cadastro_aptos"), ("% facultativo", "pc_facultativo"),
    ("idade media", "idade_media"), ("% feminino", "pc_feminino"), ("REGIAO", "regiao"),
    ("PIB per capita (R$)", "pib_per_capita"), ("quintil PIB (municipios)", "quintil_pib_mun"),
    ("ID_URNA", "id_urna"),
]
DICIONARIO = {
    "n_secoes": "seções da urna (principal + agregadas)",
    "qt_aptos": "eleitores aptos (detalhe_votacao_secao)",
    "qt_abstencao": "aptos − comparecimento",
    "Total de QT_VOTOS (validos)": "votos nominais (PT + bolsonarista); no 2º turno = votos válidos",
    "pt (% validos)": "100 × votos PT ÷ válidos",
    "pt (% aptos)": "100 × votos PT ÷ aptos; as quatro partes sobre aptos somam 100",
    "TOTAL": "eleitores do cadastro com escolaridade informada (1 a 8), somados na urna",
    "pobres": "100 × (analfabeto + lê e escreve + fundamental incompleto) ÷ TOTAL",
    "razao perfil/aptos": "eleitores do perfil (todas as escolaridades) ÷ aptos; ≠ 1 marca divergência do cadastro",
    "% facultativo": "16-17 anos, 70 ou mais e analfabetos (derivado; o perfil de 2026 não traz a coluna)",
    "PIB per capita (R$)": f"PIB municipal {ANO_PIB} (IBGE, SIDRA 5938) ÷ população do Censo 2022 (SIDRA 4714)",
    "quintil PIB (municipios)": "quintil do PIB per capita entre os 5.570 municípios (1 = mais pobre)",
}


def tabela(df):
    cols = [(rotulo, col) for rotulo, col in LAYOUT if col in df.columns]
    out = df[[col for _, col in cols]].copy()
    out.columns = [rotulo for rotulo, _ in cols]
    return out.round(4)


notas = [
    f"Presidente {ANO}, turno {TURNO}. Uma linha por urna (seção principal + agregadas).",
    f"Entram {len(analise):,} urnas; excluídas: " + "; ".join(f"{k}: {int(v):,}" for k, v in resumo["urnas"].items() if k != "(entra na análise)") + ".",
    "Aptos: QT_APTOS do detalhe_votacao_secao do TSE. Perfil: perfil_eleitor_secao do TSE, somado por urna.",
    "Resultados com dados agregados são associações condicionais (ecológicas), não comportamento individual.",
]
abas = [
    ("urnas", tabela(analise)),
    ("excluidas", tabela(base.loc[base["motivo_exclusao"] != ""]).assign(motivo=base.loc[base["motivo_exclusao"] != "", "motivo_exclusao"].to_numpy())),
    ("dicionario", pd.DataFrame({"coluna": list(DICIONARIO), "descricao": list(DICIONARIO.values())})),
    ("notas", pd.DataFrame({"nota": notas})),
]

from openpyxl import Workbook  # noqa: E402

wb = Workbook(write_only=True)
for nome, df in abas:
    ws = wb.create_sheet(nome)
    ws.append(list(df.columns))
    for ini in range(0, len(df), 50_000):
        bloco = df.iloc[ini:ini + 50_000].astype(object).where(df.iloc[ini:ini + 50_000].notna(), None)
        for linha in bloco.itertuples(index=False, name=None):
            ws.append(linha)
caminho_xlsx = SAIDA_DIR / f"urna_presidente_{SUFIXO}.xlsx"
wb.save(caminho_xlsx)
print(caminho_xlsx, f"{caminho_xlsx.stat().st_size / 1e6:.0f} MB")